#### Welcome to FabriBuddy

This notebook deploys the latest FabriBuddy into the **current workspace**. It works for the
first deployment and for every update.

[How to deploy and configure FabriBuddy](https://zan-tavcar.github.io/msfabric-fabribuddy/deploy/)

**What happens in this notebook**
- It downloads the latest FabriBuddy from GitHub.
- It creates the SQL database **FabriBuddy_DB** and stores the FabriBuddy release in it.
- It deploys or updates the notebooks and the pipeline **Load_FabriBuddy_E2E**, sorted into
  folders.
- It schedules the pipeline and runs the first scan. On a first deployment it waits for that
  scan (usually 10–30 minutes), so any problem shows up here.
- With `LIVE_MONITOR = True`, it then schedules **FabriBuddy_Live** hourly, which keeps the
  Job Monitor current minute by minute.

**Before you run it:** fill in the service principal in the next cell
([prerequisites](https://zan-tavcar.github.io/msfabric-fabribuddy/deploy/#before-you-start)).
On an update, leave it empty: the settings already in the pipeline are kept.

In [ ]:
# ── Service principal ──────────────────────────────────────────────────────────
# Its secret lives in Azure Key Vault; the notebooks read it at run time.
KEY_VAULT_URL = ""        # e.g. "https://my-vault.vault.azure.net/"
SP_TENANT_ID = ""         # Directory (tenant) ID
SP_CLIENT_ID = ""         # Application (client) ID
SP_SECRET_NAME = "fabribuddy-sp-secret"

# ── What to scan ───────────────────────────────────────────────────────────────
WORKSPACES = ""           # workspace IDs, comma-separated; empty = every workspace
TENANT_NAME = ""          # how the app names your tenant, e.g. "Contoso"

# ── Deployment ─────────────────────────────────────────────────────────────────
SCHEDULE_DAILY_AT_UTC = "05:00"   # daily scan time; "" for no schedule
RUN_AFTER_DEPLOY = True           # run a scan now (on a first deployment, wait for it)
LIVE_MONITOR = False              # Job Monitor live all day: runs FabriBuddy_Live hourly
                                  # (a small Python session, all day — it uses capacity)
REPOSITORY = "zan-tavcar/msfabric-fabribuddy"
BRANCH = "main"

### Source files

In [ ]:
import base64, json, os, re, shutil, subprocess, sys, tempfile, time, zipfile
from io import BytesIO
import requests

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "pyyaml"])
import yaml

# Download this repository (src, config, data, release) into a temporary folder.
archive = requests.get(f"https://api.github.com/repos/{REPOSITORY}/zipball/{BRANCH}", timeout=300)
archive.raise_for_status()
SOURCE = tempfile.mkdtemp(prefix="fabribuddy-")
with zipfile.ZipFile(BytesIO(archive.content)) as z:
    for name in z.namelist():
        rel = name.split("/", 1)[1] if "/" in name else ""
        if rel.startswith(("src/", "config/", "data/", "release/")) and not name.endswith("/"):
            target = os.path.join(SOURCE, rel)
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with open(target, "wb") as f:
                f.write(z.read(name))
config = yaml.safe_load(open(os.path.join(SOURCE, "config", "deployment_config.yaml")))
release = json.load(open(os.path.join(SOURCE, "data", "current_fabribuddy_version.json")))
print(f"FabriBuddy {release['version']} from {REPOSITORY}@{BRANCH}")

In [ ]:
# FabriBuddy_DB, before the FabriBuddy package is installed. As whoever runs this notebook.
import json, struct, subprocess, sys, time
import requests

FABRIC_API = "https://api.fabric.microsoft.com/v1"
WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]


def fabric(method, path, body=None, wait=True):
    """The Fabric REST API as you; waits for long-running operations."""
    headers = {"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}"}
    r = requests.request(method, f"{FABRIC_API}{path}", headers=headers, json=body, timeout=120)
    if r.status_code >= 400:
        raise RuntimeError(f"{method} {path} failed ({r.status_code}): {r.text[:400]}")
    if r.status_code == 202 and wait and r.headers.get("Location"):
        location, state = r.headers["Location"], None
        for _ in range(200):
            time.sleep(min(int(r.headers.get("Retry-After", 3) or 3), 10))
            r = requests.get(location, headers=headers, timeout=60)
            state = (r.json() if r.text else {}).get("status")
            if state in ("Succeeded", "Failed", "Undefined"):
                break
        if state != "Succeeded":
            raise RuntimeError(f"{method} {path} did not succeed: {r.text[:400]}")
        result = requests.get(location + "/result", headers=headers, timeout=60)
        return result.json() if result.status_code == 200 and result.text else {}
    return r.json() if r.text else {}


def find_sql_database(name):
    """(item id, server, database) of the SQL database called `name` here, or (None, "", "")."""
    for item in fabric("GET", f"/workspaces/{WORKSPACE_ID}/sqlDatabases").get("value", []):
        if item["displayName"].lower() == name.lower():
            props = fabric("GET", f"/workspaces/{WORKSPACE_ID}/sqlDatabases/{item['id']}").get("properties") or {}
            return item["id"], (props.get("serverFqdn") or "").split(",")[0], props.get("databaseName") or ""
    return None, "", ""


def sql_token():
    return notebookutils.credentials.getToken("https://database.windows.net/")


def sql_connect(server, database):
    import pyodbc

    raw = sql_token().encode("utf-16-le")
    return pyodbc.connect(
        f"Driver={{ODBC Driver 18 for SQL Server}};Server={server},1433;Database={database};"
        "Encrypt=yes;TrustServerCertificate=no",
        attrs_before={1256: struct.pack(f"<I{len(raw)}s", len(raw), raw)},
        timeout=30,
    )


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", *args])

In [ ]:
WORKSPACE_NAME = fabric("GET", f"/workspaces/{WORKSPACE_ID}")["displayName"]
print(f"Deploying into: {WORKSPACE_NAME}")


def item_id(name):
    """The id of item `name` ("FabriBuddy_Scan.Notebook") in this workspace, or None."""
    display, kind = name.rsplit(".", 1)
    for item in fabric("GET", f"/workspaces/{WORKSPACE_ID}/items").get("value", []):
        if item["displayName"] == display and item["type"] == kind:
            return item["id"]
    return None


def import_item(name, folder, existing_id):
    """Create item `name` from the files in `folder`, or update its definition."""
    display, kind = name.rsplit(".", 1)
    parts = []
    for root, _, files in os.walk(folder):
        for file in files:
            path = os.path.join(root, file)
            parts.append({
                "path": os.path.relpath(path, folder).replace(os.sep, "/"),
                "payload": base64.b64encode(open(path, "rb").read()).decode(),
                "payloadType": "InlineBase64",
            })
    definition = {"parts": parts}
    if kind == "Notebook":
        definition["format"] = "ipynb"
    if existing_id:
        fabric("POST", f"/workspaces/{WORKSPACE_ID}/items/{existing_id}/updateDefinition?updateMetadata=true",
               {"definition": definition})
    else:
        fabric("POST", f"/workspaces/{WORKSPACE_ID}/items",
               {"displayName": display, "type": kind, "definition": definition})

### 1. FabriBuddy_DB

In [ ]:
db_id, SQL_SERVER, SQL_DATABASE = find_sql_database(config["sql_database"])
if db_id is None:
    print(f"Creating {config['sql_database']} …")
    fabric("POST", f"/workspaces/{WORKSPACE_ID}/sqlDatabases", {
        "displayName": config["sql_database"],
        "description": "FabriBuddy: what the scan found, and what the FabriBuddy app shows.",
    })
for _ in range(60):
    db_id, SQL_SERVER, SQL_DATABASE = find_sql_database(config["sql_database"])
    if SQL_SERVER:
        break
    time.sleep(10)
if not SQL_SERVER:
    raise RuntimeError(f"{config['sql_database']} isn't ready yet. Wait a minute and run this notebook again.")
print(f"✓ {config['sql_database']}")

### 2. The FabriBuddy release

In [ ]:
wheel = os.path.join(SOURCE, release["wheel"])
pip_install(wheel)
from fabribuddy_intenant import sqldb

with sql_connect(SQL_SERVER, SQL_DATABASE) as conn:
    sqldb.ensure_schema(conn)
    sqldb.save_package(conn, os.path.basename(wheel), release["version"], open(wheel, "rb").read())
print(f"✓ FabriBuddy {release['version']} stored in {config['sql_database']}")

### 3. Notebooks and the pipeline

In [ ]:
SETTINGS = {
    "key_vault_url": KEY_VAULT_URL, "sp_tenant_id": SP_TENANT_ID, "sp_client_id": SP_CLIENT_ID,
    "sp_secret_name": SP_SECRET_NAME, "workspaces": WORKSPACES,
    "tenant_name": TENANT_NAME,
}
mapping = {config["ids"]["workspace"]: WORKSPACE_ID}


def pipeline_settings(path, existing_id):
    """Pipeline parameter defaults: kept on an update, filled from this notebook on the first deploy."""
    content = json.load(open(path))
    params = content["properties"]["parameters"]
    current = {}
    if existing_id:
        definition = fabric("POST", f"/workspaces/{WORKSPACE_ID}/items/{existing_id}/getDefinition")
        for part in definition.get("definition", {}).get("parts", []):
            if part["path"] == "pipeline-content.json":
                import base64
                current = json.loads(base64.b64decode(part["payload"]))["properties"].get("parameters", {})
    for name, spec in params.items():
        if name in current:
            spec["defaultValue"] = current[name].get("defaultValue", spec["defaultValue"])
        elif name in SETTINGS and SETTINGS[name] not in ("", None):
            spec["defaultValue"] = SETTINGS[name]
    json.dump(content, open(path, "w"), indent=2)


for name in config["items"]:
    work = tempfile.mkdtemp()
    shutil.copytree(os.path.join(SOURCE, "src", name), os.path.join(work, name))
    existing = item_id(name)
    for root, _, files in os.walk(os.path.join(work, name)):
        for file in files:
            path = os.path.join(root, file)
            text = open(path, encoding="utf-8").read()
            for old, new in mapping.items():
                text = text.replace(old, new)
            open(path, "w", encoding="utf-8").write(text)
    if name.endswith(".DataPipeline"):
        pipeline_settings(os.path.join(work, name, "pipeline-content.json"), existing)
    import_item(name, os.path.join(work, name), existing)
    new_id = item_id(name)
    mapping[config["ids"][name]] = new_id
    print(f"✓ {name} {'updated' if existing else 'created'}")

### 4. Folders

In [ ]:
for folder, names in config["folders"].items():
    found = fabric("GET", f"/workspaces/{WORKSPACE_ID}/folders").get("value", [])
    folder_id = next((f["id"] for f in found if f["displayName"] == folder), None)
    if folder_id is None:
        folder_id = fabric("POST", f"/workspaces/{WORKSPACE_ID}/folders", {"displayName": folder})["id"]
    ids = [i for i in (item_id(n) for n in names) if i]
    if ids:
        fabric("POST", f"/workspaces/{WORKSPACE_ID}/items/bulkMove", {"targetFolderId": folder_id, "items": ids})
    print(f"✓ {folder}: {', '.join(n.rsplit('.', 1)[0] for n in names)}")

### 5. Schedule and first scan

In [ ]:
import datetime as dt

pipeline_id = item_id("Load_FabriBuddy_E2E.DataPipeline")
if SCHEDULE_DAILY_AT_UTC:
    path = f"/workspaces/{WORKSPACE_ID}/items/{pipeline_id}/jobs/Pipeline/schedules"
    if fabric("GET", path).get("value"):
        print("✓ Schedule: already set (left as it is)")
    else:
        now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
        fabric("POST", path, {"enabled": True, "configuration": {
            "type": "Daily", "times": [SCHEDULE_DAILY_AT_UTC], "localTimeZoneId": "UTC",
            "startDateTime": now.strftime("%Y-%m-%dT%H:%M:%S"),
            "endDateTime": (now + dt.timedelta(days=3650)).strftime("%Y-%m-%dT%H:%M:%S"),
        }})
        print(f"✓ Schedule: Load_FabriBuddy_E2E daily at {SCHEDULE_DAILY_AT_UTC} UTC")

def scan_has_finished_before():
    """Whether any scan has saved its results in FabriBuddy_DB (false on a first deployment)."""
    with sql_connect(SQL_SERVER, SQL_DATABASE) as conn:
        return conn.cursor().execute("SELECT COUNT(*) FROM dbo.fabribuddy_state").fetchone()[0] > 0


def run_pipeline(wait):
    """Start Load_FabriBuddy_E2E; with `wait`, follow the run until it ends."""
    url = f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items/{pipeline_id}/jobs/instances?jobType=Pipeline"
    auth = lambda: {"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}"}
    r = requests.post(url, headers=auth(), timeout=120)
    if r.status_code >= 400:
        raise RuntimeError(f"Couldn't start Load_FabriBuddy_E2E ({r.status_code}): {r.text[:400]}")
    location = r.headers.get("Location")
    if not wait or not location:
        print("✓ Scan started: open Load_FabriBuddy_E2E → View run history to follow it.")
        return None
    print("Running the first scan. This usually takes 10–30 minutes; this cell waits for it.")
    started, last_note = time.time(), 0
    while time.time() - started < 3 * 3600:
        time.sleep(30)
        job = requests.get(location, headers=auth(), timeout=60).json()
        status = job.get("status")
        minutes = int((time.time() - started) / 60)
        if status == "Completed":
            print(f"✓ First scan finished in {minutes} min")
            return "Completed"
        if status in ("Failed", "Cancelled", "Deduped"):
            reason = (job.get("failureReason") or {}).get("message") or status
            raise RuntimeError(
                f"The first scan {status.lower()} after {minutes} min:\n{reason}\n\n"
                "Fix the cause (see Load_FabriBuddy_E2E → View run history for the scan's output, "
                "and https://zan-tavcar.github.io/msfabric-fabribuddy/troubleshooting/), "
                "then run this notebook again."
            )
        if minutes >= last_note + 5:
            last_note = minutes
            print(f"  … still scanning ({minutes} min)")
    print("The first scan is still running after 3 hours; not waiting any longer. "
          "Follow it in Load_FabriBuddy_E2E → View run history.")
    return None


def schedule_live():
    live_id = item_id("FabriBuddy_Live.Notebook")
    path = f"/workspaces/{WORKSPACE_ID}/items/{live_id}/jobs/RunNotebook/schedules"
    if fabric("GET", path).get("value"):
        print("✓ Live monitor: already scheduled (left as it is)")
        return
    now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0, second=0)
    fabric("POST", path, {"enabled": True, "configuration": {
        "type": "Cron", "interval": 60, "localTimeZoneId": "UTC",
        "startDateTime": now.strftime("%Y-%m-%dT%H:%M:%S"),
        "endDateTime": (now + dt.timedelta(days=3650)).strftime("%Y-%m-%dT%H:%M:%S"),
    }})
    print("✓ Live monitor: FabriBuddy_Live every hour (keeps the Job Monitor current)")


# On a first deployment, wait for the first scan: its errors show up here, and the
# live monitor only starts once there is something to keep live.
if RUN_AFTER_DEPLOY:
    run_pipeline(wait=not scan_has_finished_before())
if LIVE_MONITOR:
    schedule_live()

### 6. The FabriBuddy app

In [ ]:
print(f"""
Last step: add the FabriBuddy app from your computer, once (Node.js 20+):

  git clone https://github.com/{REPOSITORY}.git
  cd {REPOSITORY.split('/')[1]}/app
  npm install
  npx rayfin login
  npm run connect -- {WORKSPACE_ID} {db_id}
  npx rayfin up --workspace-id {WORKSPACE_ID}

Then open "fabribuddy" in this workspace.
Guide: https://zan-tavcar.github.io/msfabric-fabribuddy/deploy/#add-the-app
""")